In [90]:
!pip install -q librosa timm

import os, glob, random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [91]:
# v6: Based on v4 but optimized for 90%
# Changes: More epochs, EfficientNet-B2, slightly less noise

GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
g2i = {g: i for i, g in enumerate(GENRES)}
STEMS = ['drums', 'vocals', 'bass', 'other']
BASE = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'

SR = 22050
DUR = 10
N_MELS = 128
BATCH = 24
EPOCHS = 12  # More than v4
LR = 8e-4

songs = {g: [] for g in GENRES}
for g in GENRES:
    gd = os.path.join(BASE, 'genres_stems', g)
    if os.path.exists(gd):
        for s in os.listdir(gd):
            sp = os.path.join(gd, s)
            if os.path.isdir(sp) and all(os.path.exists(os.path.join(sp, f"{st}.wav")) for st in STEMS):
                songs[g].append(sp)
    print(f"{g}: {len(songs[g])}")

noise = glob.glob(os.path.join(BASE, 'ESC-50-master', 'audio', '*.wav'))
print(f"Noise: {len(noise)}")

blues: 100
classical: 100
country: 100
disco: 100
hiphop: 100
jazz: 100
metal: 100
pop: 100
reggae: 100
rock: 100
Noise: 2000


In [92]:
def load(p, sr=SR, d=DUR):
    try:
        a, _ = librosa.load(p, sr=sr, duration=d)
        t = sr * d
        if len(a) < t: a = np.tile(a, 3)[:t]
        return a[:t]
    except: return np.zeros(sr * d)

def mix_cross(g):
    m = np.zeros(SR * DUR, dtype=np.float32)
    for st in STEMS:
        m += load(os.path.join(random.choice(songs[g]), f"{st}.wav"))
    return m

def add_noise(a, lvl):
    n = load(random.choice(noise)) if noise else np.zeros_like(a)
    if np.max(np.abs(n)) > 0: n /= np.max(np.abs(n))
    return a + lvl * n

def norm(a):
    a = a - np.mean(a)
    if np.max(np.abs(a)) > 0: a = a / np.max(np.abs(a)) * 0.95
    return a.astype(np.float32)

def to_mel(a):
    m = librosa.feature.melspectrogram(y=a, sr=SR, n_mels=N_MELS, n_fft=2048, hop_length=512)
    m = librosa.power_to_db(m, ref=np.max)
    return (m - m.mean()) / (m.std() + 1e-6)

In [93]:
class DS(Dataset):
    def __init__(self, n=200):  # More samples
        self.d = [(g, i) for g in GENRES for i in range(n)]
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        g, _ = self.d[i]
        a = mix_cross(g)
        
        # Noise: 75% prob, level 0.1-0.35 (slightly less than v4)
        if random.random() < 0.75:
            a = add_noise(a, random.uniform(0.1, 0.35))
        
        # Time shift
        if random.random() < 0.5:
            a = np.roll(a, random.randint(-SR//2, SR//2))
        
        a = norm(a)
        m = to_mel(a)
        
        # SpecAugment (moderate)
        if random.random() < 0.5:
            t = random.randint(0, 25)
            t0 = random.randint(0, max(1, m.shape[1]-t-1))
            m[:, t0:t0+t] = 0
        if random.random() < 0.5:
            f = random.randint(0, 15)
            f0 = random.randint(0, max(1, m.shape[0]-f-1))
            m[f0:f0+f, :] = 0
        
        return torch.tensor(m, dtype=torch.float32).unsqueeze(0).repeat(3,1,1), g2i[g]

tr = DataLoader(DS(200), batch_size=BATCH, shuffle=True, num_workers=2)
print(f"Batches: {len(tr)}")

Batches: 84


In [94]:
# EfficientNet-B2 (better than B0)
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.bb = timm.create_model('efficientnet_b2', pretrained=True, num_classes=0)
        self.h = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(self.bb.num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, 10)
        )
    def forward(self, x): return self.h(self.bb(x))

model = Model().to(device)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

Params: 8,064,268


In [95]:
crit = nn.CrossEntropyLoss(label_smoothing=0.1)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(tr))

best = 0
for ep in range(EPOCHS):
    model.train()
    c, t = 0, 0
    for d, l in tqdm(tr, desc=f"Epoch {ep+1}"):
        d, l = d.to(device), l.to(device)
        opt.zero_grad()
        o = model(d)
        crit(o, l).backward()
        opt.step()
        sch.step()
        c += (o.argmax(1) == l).sum().item()
        t += l.size(0)
    
    acc = c / t
    if acc > best:
        best = acc
        torch.save(model.state_dict(), 'best.pth')
    print(f"Train Acc: {acc:.4f}, Best: {best:.4f}")

Epoch 1: 100%|██████████| 84/84 [03:17<00:00,  2.36s/it]


Train Acc: 0.2350, Best: 0.2350


Epoch 2: 100%|██████████| 84/84 [03:17<00:00,  2.36s/it]


Train Acc: 0.5745, Best: 0.5745


Epoch 3: 100%|██████████| 84/84 [03:18<00:00,  2.36s/it]


...

Train Acc: 0.9075, Best: 0.9075


Epoch 11: 100%|██████████| 84/84 [03:22<00:00,  2.41s/it]


Train Acc: 0.9045, Best: 0.9075


Epoch 12: 100%|██████████| 84/84 [03:19<00:00,  2.38s/it]


Train Acc: 0.9305, Best: 0.9305


In [96]:
# Inference with 5-crop TTA (same as v4 that worked)
model.load_state_dict(torch.load('best.pth'))
model.eval()

test_df = pd.read_csv(os.path.join(BASE, 'test.csv'))
sub = pd.read_csv(os.path.join(BASE, 'sample_submission.csv'))
i2f = dict(zip(test_df['id'], test_df['filename']))
files = [os.path.join(BASE, i2f[r['id']]) for _, r in sub.iterrows()]

def tta(path, n=5):
    try:
        af, _ = librosa.load(path, sr=SR)
    except:
        af = np.zeros(SR * 20)
    
    t = SR * DUR
    if len(af) < t: af = np.tile(af, 3)
    
    probs = []
    for p in np.linspace(0, max(0, len(af)-t), n).astype(int):
        cr = af[p:p+t]
        if len(cr) < t: cr = np.pad(cr, (0, t-len(cr)))
        cr = norm(cr)
        m = to_mel(cr)
        mt = torch.tensor(m, dtype=torch.float32).unsqueeze(0).unsqueeze(0).repeat(1,3,1,1).to(device)
        with torch.no_grad():
            probs.append(torch.softmax(model(mt), dim=1))
    return torch.stack(probs).mean(0).argmax(1).item()

print("5-crop TTA...")
preds = [tta(f) for f in tqdm(files)]

sub['genre'] = [GENRES[p] for p in preds]
sub.to_csv('submission.csv', index=False)
print("Done!")
print(sub['genre'].value_counts())

5-crop TTA...


100%|██████████| 3020/3020 [10:28<00:00,  4.80it/s]

Done!
genre
classical    342
rock         340
disco        318
reggae       305
metal        303
hiphop       301
pop          299
blues        283
jazz         280
country      249
Name: count, dtype: int64
